# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

Implemented as a **stateful directed graph**: intent classification → conditional routing → tool call (with retry + validation) → response formatting, with every step logged and appended to a trajectory for evaluation.

---
### 🛠️ What's Implemented
- Agent logic as an explicit state machine (not a flat if/else script)
- Conditional routing (rule-based intent classifier)
- Tool integration with JSON-schema input validation
- Error handling: try/except + retry loop
- **Bonus:** extra tool (word counter), structured logging, trajectory tracking, and completion-rate/cost metrics


In [1]:
import re
import json
import time
import logging
from dataclasses import dataclass, field

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger("agent")


## 🛠️ Tools
Each tool exposes a JSON schema for its expected input, so the agent can validate arguments before calling it (Quiz Q6).

In [2]:
TOOL_SCHEMAS = {
    "calculator": {
        "type": "object",
        "properties": {"expression": {"type": "string"}},
        "required": ["expression"]
    },
    "keyword_extractor": {
        "type": "object",
        "properties": {"text": {"type": "string"}},
        "required": ["text"]
    },
    "word_counter": {
        "type": "object",
        "properties": {"text": {"type": "string"}},
        "required": ["text"]
    }
}

def validate_input(tool_name, payload):
    schema = TOOL_SCHEMAS[tool_name]
    for field_name, spec in schema["properties"].items():
        if field_name in payload and not isinstance(payload[field_name], str) and spec["type"] == "string":
            raise ValueError(f"'{field_name}' must be a string for {tool_name}")
    for required_field in schema["required"]:
        if required_field not in payload or not payload[required_field]:
            raise ValueError(f"Missing required field '{required_field}' for {tool_name}")


In [3]:
SAFE_EXPR_PATTERN = re.compile(r"^[0-9\.\+\-\*\/\(\)\s]+$")

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression safely (digits/operators only, no eval on arbitrary input)."""
    validate_input("calculator", {"expression": expression})
    if not SAFE_EXPR_PATTERN.match(expression):
        raise ValueError("Expression contains disallowed characters")
    return str(eval(expression, {"__builtins__": {}}, {}))


In [4]:
def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    validate_input("keyword_extractor", {"text": text})
    words = text.split()
    keywords = list(set(w.lower().strip(".,!?") for w in words if len(w) > 4))
    return keywords[:5]


In [5]:
def word_counter(text: str) -> dict:
    """Bonus tool: return word and character counts."""
    validate_input("word_counter", {"text": text})
    words = text.split()
    return {"word_count": len(words), "char_count": len(text)}


## 🤖 Agent Logic

**Intent classifier** — conditional routing rules (Quiz Q3):
- contains "calculate" → `calculator`
- contains "keyword" → `keyword_extractor`
- contains "count words" / "word count" → `word_counter` (bonus)
- else → general response

**Trajectory** — every node the query passes through is recorded (Quiz Q9), so we can inspect *how* the agent reached an answer, not just the final output.

**Retry loop** — tool calls get up to `MAX_RETRIES` attempts before the agent gives up and returns a structured error (Quiz Q4, Q8).


In [6]:
MAX_RETRIES = 2

@dataclass
class AgentState:
    query: str
    trajectory: list = field(default_factory=list)
    tool_calls: int = 0

    def log_step(self, node, detail):
        self.trajectory.append({"node": node, "detail": detail})
        logger.info(f"[{node}] {detail}")


In [7]:
def classify_intent(query_lower: str) -> str:
    if "calculate" in query_lower or re.search(r"\d+\s*[\+\-\*/]\s*\d+", query_lower):
        return "calculator"
    if "keyword" in query_lower:
        return "keyword_extractor"
    if "count" in query_lower and "word" in query_lower:
        return "word_counter"
    return "general"


In [8]:
def run_tool_with_retry(tool_fn, state, **kwargs):
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            result = tool_fn(**kwargs)
            state.tool_calls += 1
            state.log_step(tool_fn.__name__, f"attempt {attempt} succeeded")
            return result, None
        except Exception as e:
            last_error = str(e)
            state.log_step(tool_fn.__name__, f"attempt {attempt} failed: {last_error}")
    return None, last_error


In [9]:
def agent(query: str) -> dict:
    state = AgentState(query=query)
    query_lower = query.lower()
    state.log_step("intent_classifier", f"query received: '{query}'")

    intent = classify_intent(query_lower)
    state.log_step("router", f"routed to '{intent}'")

    if intent == "calculator":
        match = re.search(r"[\d\.\+\-\*/\(\)\s]+", query)
        expression = match.group().strip() if match else query
        result, error = run_tool_with_retry(calculator, state, expression=expression)
        if error:
            return finalize(state, "error", f"Calculator failed: {error}")
        return finalize(state, "calculation", result)

    if intent == "keyword_extractor":
        result, error = run_tool_with_retry(extract_keywords, state, text=query)
        if error:
            return finalize(state, "error", f"Keyword extraction failed: {error}")
        return finalize(state, "keywords", result)

    if intent == "word_counter":
        result, error = run_tool_with_retry(word_counter, state, text=query)
        if error:
            return finalize(state, "error", f"Word counter failed: {error}")
        return finalize(state, "word_count", result)

    state.log_step("general_response", "no tool needed")
    return finalize(state, "general", f"I don't have a specific tool for that, but here's your query noted: '{query}'")


def finalize(state: AgentState, result_type: str, result) -> dict:
    state.log_step("response_formatter", f"final type='{result_type}'")
    return {
        "type": result_type,
        "result": result,
        "trajectory": state.trajectory,
        "tool_calls": state.tool_calls
    }


## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / word_count / general / error",
  "result": ...,
  "trajectory": [...],
  "tool_calls": <int>
}
```

In [10]:
queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?",
    "count words in this sentence please",
    "Calculate 10 / 0"
]

for q in queries:
    print("Query:", q)
    response = agent(q)
    print("Type:", response["type"])
    print("Result:", response["result"])
    print("Tool calls:", response["tool_calls"])
    print("-" * 50)


Query: Calculate 20 + 5
Type: calculation
Result: 25
Tool calls: 1
--------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Type: keywords
Result: ['industries', 'artificial', 'intelligence', 'keywords', 'extract']
Tool calls: 1
--------------------------------------------------
Query: What is machine learning?
Type: general
Result: I don't have a specific tool for that, but here's your query noted: 'What is machine learning?'
Tool calls: 0
--------------------------------------------------
Query: count words in this sentence please
Type: word_count
Result: {'word_count': 6, 'char_count': 35}
Tool calls: 1
--------------------------------------------------
Query: Calculate 10 / 0
Type: error
Result: Calculator failed: division by zero
Tool calls: 0
--------------------------------------------------


## 📊 Evaluation Metrics (Quiz Q10)
Task completion rate = successful responses / total queries. Cost = total tool calls across all queries (a simple proxy for API/compute cost).

In [11]:
def evaluate(queries_list):
    total = len(queries_list)
    completed = 0
    total_tool_calls = 0

    for q in queries_list:
        response = agent(q)
        total_tool_calls += response["tool_calls"]
        if response["type"] != "error":
            completed += 1

    return {
        "task_completion_rate": round(completed / total, 2) if total else 0,
        "total_tool_calls": total_tool_calls,
        "avg_tool_calls_per_query": round(total_tool_calls / total, 2) if total else 0
    }

evaluate(queries)


{'task_completion_rate': 0.8,
 'total_tool_calls': 3,
 'avg_tool_calls_per_query': 0.6}

## 🎯 Interactive Mode

In [12]:
while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", agent(user_input))


Enter query (type 'exit' to stop): exit
